In [2]:

# Titanic Survival Prediction - Mini Project
# Author: Student Version (Original Implementation)

import pandas as pd
import numpy as np
import re
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("titanic.csv")

print("Dataset Shape:", df.shape)
print(df.head())

# =========================
# 2. FEATURE ENGINEERING
# =========================

# Extract title from passenger name
def extract_title(name):
    match = re.search(r',\s*([^\.]+)\.', str(name))
    if match:
        return match.group(1).strip()
    return "Unknown"

df["Title"] = df["Name"].apply(extract_title)

# Family size
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Cabin availability
df["CabinPresent"] = df["Cabin"].notna().astype(int)

# Age groups (optional feature)
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
)

# =========================
# 3. DATA PREPARATION
# =========================

target = "Survived"

drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]

X = df.drop(columns=[target] + drop_cols)
y = df[target]

numeric_features = [
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "FamilySize"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Pclass",
    "Title",
    "AgeGroup",
    "CabinPresent"
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# =========================
# 4. TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =========================
# 5. MODEL BUILDING
# =========================

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ]
)

pipeline.fit(X_train, y_train)

# =========================
# 6. EVALUATION
# =========================

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, y_prob), 4))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

# =========================
# 7. FEATURE IMPORTANCE
# =========================

feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

importances = pipeline.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print("\nTop 15 Important Features")
print(importance_df.head(15))

# =========================
# 8. SAVE MODEL
# =========================

joblib.dump(pipeline, "titanic_survival_model.pkl")
print("\nModel saved as titanic_survival_model.pkl")

# =========================
# 9. INFERENCE EXAMPLE
# =========================

sample = X.iloc[[0]]
prediction = pipeline.predict(sample)[0]
probability = pipeline.predict_proba(sample)[0][1]

print("\nInference Example")
print("Predicted Survival:", prediction)
print("Survival Probability:", round(probability, 4))


Dataset Shape: (418, 12)
   PassengerId  Survived  Pclass  \
0          892         0       3   
1          893         1       3   
2          894         0       2   
3          895         0       3   
4          896         1       3   

                                           Name     Sex   Age  SibSp  Parch  \
0                              Kelly, Mr. James    male  34.5      0      0   
1              Wilkes, Mrs. James (Ellen Needs)  female  47.0      1      0   
2                     Myles, Mr. Thomas Francis    male  62.0      0      0   
3                              Wirz, Mr. Albert    male  27.0      0      0   
4  Hirvonen, Mrs. Alexander (Helga E Lindqvist)  female  22.0      1      1   

    Ticket     Fare Cabin Embarked  
0   330911   7.8292   NaN        Q  
1   363272   7.0000   NaN        S  
2   240276   9.6875   NaN        Q  
3   315154   8.6625   NaN        S  
4  3101298  12.2875   NaN        S  

Accuracy: 1.0
ROC-AUC : 1.0

Classification Report
         